In [19]:
import pandas as pd
import numpy as np
import ast

movies = pd.read_csv('../data/tmdb_5000_movies.csv')
credits = pd.read_csv('../data/tmdb_5000_credits.csv')
credits.columns = ['movie_id', 'title', 'cast', 'crew']

df = movies.merge(credits, left_on='id', right_on='movie_id')
df = df.drop(columns=['title_y', 'movie_id'])
df = df.rename(columns={'title_x': 'title'})
df.shape

(4803, 22)

In [20]:
import ast

def get_list(x):
    L = ast.literal_eval(x)
    names = [i['name'] for i in L]
    return names

df['genres_list'] = df['genres'].apply(get_list)
df['keywords_list'] = df['keywords'].apply(get_list)
df['cast_list'] = df['cast'].apply(get_list)
df['cast_list'] = df['cast_list'].apply(lambda x: x[:3])

def get_director(x):
    crew = ast.literal_eval(x)
    for person in crew:
        if person['job'] == 'Director':
            return person['name']
    return np.nan

df['director'] = df['crew'].apply(get_director)
df[['title','genres_list','cast_list','director']].head()

,title,genres_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[Sam Worthington, Zoe Saldana, Sigourney Weaver]",James Cameron
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[Johnny Depp, Orlando Bloom, Keira Knightley]",Gore Verbinski
2,Spectre,"[Action, Adventure, Crime]","[Daniel Craig, Christoph Waltz, Léa Seydoux]",Sam Mendes
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[Christian Bale, Michael Caine, Gary Oldman]",Christopher Nolan
4,John Carter,"[Action, Adventure, Science Fiction]","[Taylor Kitsch, Lynn Collins, Samantha Morton]",Andrew Stanton


In [21]:
def clean_names(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        return ''

for col in ['genres_list', 'keywords_list', 'cast_list']:
    df[col] = df[col].apply(clean_names)

df['director_clean'] = df['director'].apply(lambda x: str.lower(x.replace(" ","")) if isinstance(x, str) else '')

df['tags'] = df['genres_list'] + df['keywords_list'] + df['cast_list']
df['tags'] = df['tags'].apply(lambda x: ' '.join(x)) + ' ' + df['director_clean']

from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(df['tags'])
matrix.shape

(4803, 17832)

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['tags'])

print(tfidf_matrix.shape)  

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(cosine_sim.shape)  

(4803, 30900)
(4803, 4803)


In [22]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(matrix)

def recommend(title):
    idx = df[df['title'] == title].index[0]
    scores = list(enumerate(sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = scores[1:6]
    
    movies = []
    for i in scores:
        movies.append(df.iloc[i[0]]['title'])
    return movies

recommend('Avatar')

['Star Trek Into Darkness',
 'Aliens',
 'Alien³',
 'Predators',
 'Jupiter Ascending']

In [23]:
import numpy as np
np.random.seed(42)

n_users = 500
ratings_per_user = 20

genres_all = df['genres_list'].explode().dropna().unique()

data = []
for u in range(n_users):
    fav = np.random.choice(genres_all, 2, replace=False)
    subset = df[df['genres_list'].apply(lambda g: any(x in g for x in fav))]
    if len(subset) < ratings_per_user:
        subset = df
    picks = subset.sample(ratings_per_user)
    for _, row in picks.iterrows():
        r = np.random.randint(3,6)
        data.append([u, row['id'], r])

ratings_df = pd.DataFrame(data, columns=['user_id','movie_id','rating'])
ratings_df.head()

,user_id,movie_id,rating
0,0,2320,4
1,0,10431,4
2,0,45138,3
3,0,1735,5
4,0,5174,3


In [24]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(ratings_df, reader)
train, test = train_test_split(data, test_size=0.2)

model = SVD()
model.fit(train)
preds = model.test(test)
accuracy.rmse(preds)

RMSE: 0.8396


np.float64(0.8395761128395882)

In [26]:
def hybrid(title, user_id, n=5, w=0.5):
    idx = indices[title]
    scores = list(enumerate(sim[idx]))
    
    results = []
    for i, content_score in scores:
        if i == idx:
            continue
        mid = df.iloc[i]['id']
        collab_score = model.predict(user_id, mid).est / 5
        final = w*content_score + (1-w)*collab_score
        results.append((i, final))
    
    results.sort(key=lambda x: x[1], reverse=True)
    top = results[:n]
    return [df.iloc[i]['title'] for i,_ in top]

print(hybrid('Avatar', 1))
print(hybrid('Avatar', 2))

['Aliens', 'Star Trek Into Darkness', 'Alien³', 'Split Second', 'A Monster in Paris']
['Star Trek Into Darkness', 'Aliens', 'Predator', 'Jupiter Ascending', 'Guardians of the Galaxy']


In [28]:
df['profit_ratio'] = df['revenue'] / df['budget']
df['hit'] = (df['profit_ratio'] >= 2).astype(int)

df_model = df[df['budget'] > 0].copy()
df_model['hit'].value_counts()

hit
0    1957
1    1809
Name: count, dtype: int64

In [30]:
df['ratio'] = df['revenue'] / df['budget']
df['hit'] = (df['ratio'] >= 2).astype(int)

d = df[df['budget'] > 0].copy()
d['release_date'] = pd.to_datetime(d['release_date'], errors='coerce')
d['month'] = d['release_date'].dt.month
d['genre1'] = d['genres_list'].apply(lambda x: x[0] if len(x)>0 else 'Unknown')

dummies = pd.get_dummies(d['genre1'], prefix='g')
X = pd.concat([d[['budget','runtime','popularity','month']], dummies], axis=1)
X = X.fillna(X.median())
y = d['hit']

X.shape

(3766, 25)

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train, y_train)

pred = rf.predict(X_test)
print(accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

0.7015915119363395
              precision    recall  f1-score   support

           0       0.73      0.73      0.73       415
           1       0.67      0.67      0.67       339

    accuracy                           0.70       754
   macro avg       0.70      0.70      0.70       754
weighted avg       0.70      0.70      0.70       754



In [32]:
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp.sort_values(ascending=False).head(10)

popularity     0.408244
budget         0.207554
runtime        0.174322
month          0.101322
g_comedy       0.013344
g_action       0.011965
g_drama        0.011641
g_horror       0.011351
g_adventure    0.010232
g_thriller     0.006462
dtype: float64

In [33]:
X2 = X.drop(columns=['popularity'])

X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y, test_size=0.2, random_state=1)

rf2 = RandomForestClassifier(n_estimators=100, random_state=1)
rf2.fit(X_train2, y_train2)

pred2 = rf2.predict(X_test2)
print(accuracy_score(y_test2, pred2))
print(classification_report(y_test2, pred2))

0.5397877984084881
              precision    recall  f1-score   support

           0       0.59      0.56      0.57       415
           1       0.49      0.52      0.50       339

    accuracy                           0.54       754
   macro avg       0.54      0.54      0.54       754
weighted avg       0.54      0.54      0.54       754



In [34]:
imp2 = pd.Series(rf2.feature_importances_, index=X2.columns)
imp2.sort_values(ascending=False).head(10)

budget         0.394809
runtime        0.341560
month          0.162975
g_horror       0.013335
g_drama        0.009219
g_action       0.008779
g_comedy       0.008560
g_adventure    0.007805
g_thriller     0.006572
g_crime        0.005587
dtype: float64